# Jour 5 - Classification : `high_engagement`
Cible construite à partir de `vote_count` (seuil fixe 4 000). `vote_count` n'est jamais une feature.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
%matplotlib inline
from sklearn.model_selection import train_test_split
import src.models.classification as C

## 1. Données et cible

In [ ]:
df = C.load_dataset(C.THRESHOLD)
print(df.shape)
df[C.TARGET].value_counts(normalize=True).round(3)

## 2. Variables : numériques, binaires, catégorielles, texte

In [ ]:
groups = C.get_feature_groups(df)
feature_cols = groups['numeric'] + groups['binary'] + groups['categorical'] + [C.TEXT_COL]
C.assert_no_leakage(feature_cols)   # lève une erreur si vote_count & co. se glissent dans les features
{k: len(v) for k, v in groups.items()}

## 3. Split stratifié
La stratification garde la même proportion de classes dans le train et le test.

In [ ]:
X, y = df[feature_cols], df[C.TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=C.TEST_SIZE, stratify=y, random_state=C.RANDOM_STATE)
print(len(X_train), len(X_test), y_train.mean().round(3), y_test.mean().round(3))

## 4. Le Pipeline (ColumnTransformer + modèle)

In [ ]:
pipe = C.build_pipeline(C.get_models()['Logistic Regression'], groups)
pipe

## 5. Comparaison des 3 modèles + baseline

In [ ]:
results, fitted, preds = C.run_models(X_train, X_test, y_train, y_test, groups)
results

## 6. Matrices de confusion et courbes ROC

In [ ]:
C.plot_confusions(preds, y_test, show=True)
C.plot_roc(fitted, X_test, y_test, show=True)

## 7. Variables influentes

In [ ]:
C.plot_importances(fitted, show=True)

## 8. Démonstration de leakage
On ajoute `popularity`, `vote_average` et `revenue` : le score monte, mais il est artificiel.

In [ ]:
g2 = {**groups, 'numeric': groups['numeric'] + C.PROXY_FEATURES}
cols2 = g2['numeric'] + groups['binary'] + groups['categorical'] + [C.TEXT_COL]
Xa, Xb, ya, yb = train_test_split(df[cols2], y, test_size=C.TEST_SIZE, stratify=y, random_state=C.RANDOM_STATE)
leak, _, _ = C.run_models(Xa, Xb, ya, yb, g2, with_baseline=False)
leak[['model','accuracy','f1','roc_auc']]

## 9. Sauvegarde complète
(ou `python src/models/classification.py`)

In [ ]:
info = C.run_classification()
info['results']